In [1]:
import sys
import random
from collections import Counter
from pathlib import Path

sys.path.append("../src")

from mortgage_agent.schemas import Applicant
from mortgage_agent.generator import (
    make_loan,
    make_debts,
    make_income,
    make_clean_applicant,
    TRAPS,
    make_applicant,
    generate_dataset,
    save_dataset,
)

In [2]:
rng = random.Random(42)
for _ in range(3):
    print(make_loan(rng))

print()
rng = random.Random(42)
print(make_loan(rng))

amount=212000.0 property_value=265000.0 monthly_housing_payment=1705.0 estimated_closing_costs=6029.0 property_type='2-4 unit' occupancy='principal residence' underwriting='DU'
amount=504000.0 property_value=720000.0 monthly_housing_payment=4005.0 estimated_closing_costs=19272.0 property_type='1-unit' occupancy='principal residence' underwriting='DU'
amount=244000.0 property_value=305000.0 monthly_housing_payment=1947.0 estimated_closing_costs=7387.0 property_type='1-unit' occupancy='second home' underwriting='DU'

amount=212000.0 property_value=265000.0 monthly_housing_payment=1705.0 estimated_closing_costs=6029.0 property_type='2-4 unit' occupancy='principal residence' underwriting='DU'


In [3]:
rng = random.Random(5)
loan = make_loan(rng)
debts = make_debts(rng)
income = make_income(rng, loan.monthly_housing_payment, debts)

print(loan.monthly_housing_payment)
for d in debts:
    print(d)
print(income)

total = loan.monthly_housing_payment + sum(d.monthly_payment for d in debts)
print(f"DTI: {total / income.verified_monthly_income * 100:.1f}%")

4233.0
kind='credit card' monthly_payment=105.0 balance=2100.0 months_remaining=None
kind='student loan' monthly_payment=300.0 balance=15900.0 months_remaining=53
employer='Northwind Logistics' monthly_income=18200.0 verified_monthly_income=18200.0 years_employed=3.5 pay_frequency='monthly' self_employed=False
DTI: 25.5%


In [4]:
from collections import Counter

rng = random.Random(1)
a = make_clean_applicant(rng, "C001")
print(a.name, "|", a.loan.occupancy, "|", a.credit.score, "|", a.assets.bank_balance, "|", a.expected.decision)

rng = random.Random(2026)
results = Counter(make_clean_applicant(rng, f"T{i}").expected.decision for i in range(1000))
print(results)

Mei Smith | principal residence | 769 | 327000.0 | approve
Counter({'approve': 1000})


In [5]:
rng = random.Random(2026)
for trap in TRAPS:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:20} {dict(results)}")

print()
rng = random.Random(3)
a = make_applicant(rng, "A001", "recent_bankruptcy")
print(a.closing_date, a.credit.bankruptcy_chapter, a.credit.bankruptcy_discharge_date)
print(a.expected)

none                 {'approve': 500}
low_score            {'decline': 500}
high_dti             {'decline': 500}
short_debt           {'approve': 500}
recent_bankruptcy    {'decline': 500}
old_bankruptcy       {'approve': 500}
recent_foreclosure   {'decline': 500}
old_foreclosure      {'approve': 500}
deposit_needed       {'approve with conditions': 500}
deposit_not_needed   {'approve': 500}
short_funds          {'decline': 500}
gift_ok              {'approve': 500}
gift_no_letter       {'approve with conditions': 500}
gift_bad_donor       {'decline': 500}
gift_rental          {'decline': 500}
min_contribution_short {'decline': 500}
min_contribution_met {'approve': 500}

2026-12-06 13 2024-12-14
decision='decline' reasons=['Chapter 13 bankruptcy discharged 2024-12-14; the 2-year wait ends 2026-12-14, after the 2026-12-06 closing.'] rule_ids=['B3-5.3-07']


In [6]:
rng = random.Random(2026)
for trap in ["deposit_needed", "deposit_not_needed", "short_funds"]:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:20} {dict(results)}")

print()
rng = random.Random(4)
a = make_applicant(rng, "A001", "deposit_needed")
print(a.income.verified_monthly_income, a.assets.bank_balance, a.assets.large_deposits[0].amount)
print(a.expected)

deposit_needed       {'approve with conditions': 500}
deposit_not_needed   {'approve': 500}
short_funds          {'decline': 500}

7000.0 90448 22000.0
decision='approve with conditions' reasons=['Verified funds of $68,448 are short of the $70,548 needed; document the source of $22,000 in large deposits.'] rule_ids=['B3-4.2-02']


In [7]:
rng = random.Random(2026)
gift_traps = ["gift_ok", "gift_no_letter", "gift_bad_donor", "gift_rental", "min_contribution_short", "min_contribution_met"]
for trap in gift_traps:
    results = Counter(make_applicant(rng, "T", trap).expected.decision for _ in range(500))
    print(f"{trap:24} {dict(results)}")

print()
rng = random.Random(8)
a = make_applicant(rng, "A001", "min_contribution_short")
print(a.loan.property_value, a.assets.bank_balance, a.assets.gifts[0].amount)
print(a.expected)

gift_ok                  {'approve': 500}
gift_no_letter           {'approve with conditions': 500}
gift_bad_donor           {'decline': 500}
gift_rental              {'decline': 500}
min_contribution_short   {'decline': 500}
min_contribution_met     {'approve': 500}

490000.0 13400 54300.0
decision='decline' reasons=["Gifts are used on a second home at 90% LTV, so 5% ($24,500) must come from the borrower's own funds; they have $13,400."] rule_ids=['B3-4.3-04']


In [8]:
dataset = generate_dataset()
print(len(dataset))
print(Counter(a.expected.decision for _, a in dataset))
for trap, a in dataset[:3]:
    print(a.applicant_id, a.name, "|", trap, "|", a.expected.decision)

folder = Path("../data/applicants/generated")
save_dataset(dataset, folder)
print(len(list(folder.glob("G*.json"))))

loaded = Applicant.model_validate_json((folder / "G001.json").read_text())
print(loaded == dataset[0][1])

again = generate_dataset()
print(all(a1 == a2 for (_, a1), (_, a2) in zip(dataset, again)))

119
Counter({'decline': 56, 'approve': 49, 'approve with conditions': 14})
G001 Diego Haddad | recent_bankruptcy | decline
G002 Diego Nguyen | short_debt | approve
G003 Liam Brown | none | approve
119
True
True
